<a href="https://colab.research.google.com/github/DKavya8/chestxray-bias-audit/blob/main/notebooks/09_disease_level_decomposition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pyarrow
import numpy as np, pandas as pd, glob, os

In [2]:
!rm -rf /content/repo && git clone -q https://github.com/DKavya8/chestxray-bias-audit /content/repo
REPO = "/content/repo"
RESULTS_DIR = f"{REPO}/results/group_b_densenet"
print("files:", os.listdir(RESULTS_DIR))

files: ['group_b_densenet_agestd_with_ci.csv', 'run_manifest.json', 'group_b_summary_across_splits.csv', 'group_b_patient_level (1).parquet', 'group_b_densenet_agestd_points.csv', 'group_b_counts_by_finding (1).parquet']


In [3]:
pl = pd.read_parquet(f"{RESULTS_DIR}/group_b_patient_level (1).parquet")
c  = pd.read_parquet(f"{RESULTS_DIR}/group_b_counts_by_finding (1).parquet")
thr = (c[c.condition=="raw"][["split_seed","finding","threshold"]]
       .drop_duplicates().set_index(["split_seed","finding"])["threshold"])
splits = sorted(pl.split_seed.unique())
FINDINGS = ["Atelectasis","Consolidation","Infiltration","Pneumothorax","Edema","Emphysema",
            "Fibrosis","Effusion","Pneumonia","Pleural_Thickening","Cardiomegaly","Nodule","Mass","Hernia"]
print("patient rows:", len(pl), "| splits:", len(splits))

patient rows: 61594 | splits: 10


In [4]:
def fnr_weighted(is_fn, w):
    tot = w.sum()
    return (w[is_fn].sum()/tot) if tot > 0 else np.nan

def per_split_stats(d, seed, idx=None):
    dd = d if idx is None else d.iloc[idx]
    sex = dd["sex"].values
    out = {}
    for f in FINDINGS:
        t = thr.loc[(seed, f)]
        y = dd["y_"+f].values; pred = (dd["s_"+f].values >= t).astype(int)
        fn = (y==1) & (pred==0); pos = (y==1)
        g = {}
        for cond, wcol in [("raw",None),("match","matched_frac"),("ipw","ipw_weight")]:
            w = np.ones(len(dd)) if wcol is None else dd[wcol].values
            gf = fnr_weighted(fn[pos & (sex=="F")], w[pos & (sex=="F")])
            gm = fnr_weighted(fn[pos & (sex=="M")], w[pos & (sex=="M")])
            g[cond] = gf - gm
        out[f] = (g["raw"], g["match"], g["ipw"], g["raw"]-g["match"], g["raw"]-g["ipw"])
    return out

In [5]:
cache = {s: pl[pl.split_seed==s].reset_index(drop=True) for s in splits}
point = {f: np.zeros(5) for f in FINDINGS}
for s in splits:
    st = per_split_stats(cache[s], s)
    for f in FINDINGS: point[f] += np.array(st[f])/len(splits)
print("point estimates computed")

point estimates computed


In [6]:
B = 1000
rng = np.random.default_rng(20260823)
pmap = {s: cache[s].groupby("Patient ID").indices for s in splits}
draws = {f: np.full((B,5), np.nan) for f in FINDINGS}
for b in range(B):
    if b % 200 == 0: print(f"draw {b}/{B}")
    acc = {f: np.zeros(5) for f in FINDINGS}
    for s in splits:
        pids = np.array(list(pmap[s].keys()))
        samp = rng.choice(pids, size=len(pids), replace=True)
        idx = np.concatenate([pmap[s][p] for p in samp])
        st = per_split_stats(cache[s], s, idx)
        for f in FINDINGS: acc[f] += np.array(st[f])/len(splits)
    for f in FINDINGS: draws[f][b] = acc[f]
print("bootstrap done")

draw 0/1000
draw 200/1000
draw 400/1000
draw 600/1000
draw 800/1000
bootstrap done


In [7]:
METRICS = ["S_raw","S_match","S_ipw","dS_match","dS_ipw"]
# BH on dS_ipw (index 4)
p = []
for f in FINDINGS:
    col = draws[f][:,4]; share_le0 = np.mean(col<=0)
    p.append(min(1.0, 2*min(share_le0, 1-share_le0)))
p = np.array(p); order = np.argsort(p); m = len(p)
crit = (np.arange(1,m+1)/m)*0.05
passed = p[order] <= crit
kmax = np.where(passed)[0].max()+1 if passed.any() else 0
sig = np.zeros(m, bool)
if kmax>0: sig[order[:kmax]] = True
bh = {f:(round(float(p[i]),4), bool(sig[i])) for i,f in enumerate(FINDINGS)}

rows = []
for f in FINDINGS:
    lo = np.nanpercentile(draws[f],2.5,axis=0); hi = np.nanpercentile(draws[f],97.5,axis=0)
    for i,lab in enumerate(METRICS):
        rows.append(dict(dataset="NIH", backbone="densenet121-res224-all", finding=f, metric=lab,
                         value=round(float(point[f][i]),6),
                         ci_lower=round(float(lo[i]),6), ci_upper=round(float(hi[i]),6)))
res = pd.DataFrame(rows)
res.to_csv(f"{RESULTS_DIR}/group_b_disease_level_with_ci.csv", index=False)

piv = (res[res.metric.isin(["S_raw","S_ipw","dS_ipw"])]
       .pivot(index="finding", columns="metric", values="value").sort_values("S_raw", ascending=False))
piv["dS_ipw_p_BH"] = [f"{bh[f][0]:.3f}{'*' if bh[f][1] else ''}" for f in piv.index]
print(piv.to_string())
print("\nwrote group_b_disease_level_with_ci.csv  (* = significant after Benjamini-Hochberg)")

metric                 S_ipw     S_raw    dS_ipw dS_ipw_p_BH
finding                                                     
Nodule              0.118245  0.119969  0.001724       0.288
Pneumothorax        0.097464  0.110352  0.012888      0.000*
Infiltration        0.060203  0.067996  0.007793      0.000*
Mass                0.056509  0.060163  0.003654      0.032*
Pleural_Thickening  0.007273  0.017953  0.010680      0.000*
Emphysema          -0.002751  0.016843  0.019594      0.000*
Cardiomegaly       -0.006769 -0.000991  0.005778      0.008*
Pneumonia          -0.010399 -0.002484  0.007915       0.080
Atelectasis        -0.014030 -0.004091  0.009939      0.000*
Fibrosis           -0.037562 -0.017482  0.020080      0.000*
Effusion           -0.036206 -0.029239  0.006967      0.000*
Consolidation      -0.053984 -0.040075  0.013909      0.000*
Hernia             -0.210847 -0.189447  0.021400      0.002*
Edema              -0.254484 -0.268696 -0.014212       0.078

wrote group_b_disease_l